# 06. Module B - ML Distractor Generation

This notebook trains a Machine Learning ranker (Random Forest) to select the best distractors for a generated question.
It uses **TF-IDF Cosine Similarity**, **Character-level Match Score**, and **Passage Frequency** as features.


In [1]:
import os
import random
import numpy as np
import pandas as pd
import joblib
import nltk
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.metrics.pairwise import cosine_similarity
from nltk.tokenize import word_tokenize

nltk.download('punkt_tab', quiet=True)

PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models_new")

print("Loading data and TF-IDF vectorizer...")
train_df = pd.read_csv(PROCESSED_DIR / "train_verification.csv")
tfidf = joblib.load(MODELS_DIR / "tfidf_vectorizer.pkl")


Loading data and TF-IDF vectorizer...


In [2]:
def char_jaccard(str1, str2):
    set1 = set(str1.lower())
    set2 = set(str2.lower())
    union = len(set1 | set2)
    if union == 0: return 0.0
    return len(set1 & set2) / union

def get_ngrams(text, n=2):
    tokens = word_tokenize(str(text))
    ngrams = []
    for i in range(len(tokens)-n+1):
        ngrams.append(" ".join(tokens[i:i+n]))
    return ngrams

def extract_candidates(article):
    """Extract frequent 1, 2, and 3-grams from passage as candidates."""
    article = str(article).lower()
    unigrams = article.split()
    bigrams = get_ngrams(article, 2)
    trigrams = get_ngrams(article, 3)
    
    all_cands = unigrams + bigrams + trigrams
    freq = {}
    for c in all_cands:
        if len(c) > 2:
            freq[c] = freq.get(c, 0) + 1
            
    # Return unique candidates
    return list(freq.keys())

print("Functions defined.")


Functions defined.


In [3]:
# Build Training Dataset for Distractor Ranker
# To save memory, we'll subsample the training set to just 5000 unique questions
print("Building Distractor Training Dataset...")

# Group by sample_id to reconstruct MCQs
mcq_groups = train_df.groupby('sample_id', sort=False)

X_dist = []
y_dist = []

count = 0
for sample_id, group in mcq_groups:
    if count >= 3000:
        break
        
    correct_row = group[group['label'] == 1]
    if len(correct_row) == 0: continue
        
    article = str(correct_row.iloc[0]['article'])
    correct_answer = str(correct_row.iloc[0]['option_text'])
    
    distractor_rows = group[group['label'] == 0]
    good_distractors = distractor_rows['option_text'].tolist()
    
    # Positive Examples (Label = 1, these are real human-made distractors)
    for dist in good_distractors:
        X_dist.append((article, correct_answer, str(dist)))
        y_dist.append(1)
        
    # Negative Examples (Label = 0, random n-grams from text that aren't the answer)
    cands = extract_candidates(article)
    bad_distractors = [c for c in cands if c not in good_distractors and c not in correct_answer and correct_answer not in c]
    random.shuffle(bad_distractors)
    
    for dist in bad_distractors[:5]: # Take 5 random bad distractors per MCQ
        X_dist.append((article, correct_answer, str(dist)))
        y_dist.append(0)
        
    count += 1
    
print(f"Generated {len(X_dist)} training rows.")


Building Distractor Training Dataset...


Generated 24000 training rows.


In [4]:
print("Extracting Features for Distractor Model...")

# We extract: 
# 1. TF-IDF Cosine Similarity to Correct Answer
# 2. Character-level match score
# 3. Passage frequency

X_features = []

# Pre-transform articles and answers to speed things up
articles = [r[0] for r in X_dist]
answers = [r[1] for r in X_dist]
candidates = [r[2] for r in X_dist]

print("  Transforming strings...")
ans_mat = tfidf.transform(answers)
cand_mat = tfidf.transform(candidates)

print("  Computing cosine similarities...")
# Element-wise dot product of sparse matrices
sim_scores = ans_mat.multiply(cand_mat).sum(axis=1).A1

print("  Computing character match and frequency...")
for i in range(len(X_dist)):
    art = articles[i].lower()
    ans = answers[i]
    cand = candidates[i]
    
    sim = sim_scores[i]
    char_match = char_jaccard(ans, cand)
    freq = art.count(cand.lower())
    
    X_features.append([sim, char_match, freq])

X_features = np.array(X_features)
y_dist = np.array(y_dist)

print("Feature extraction complete. Shape:", X_features.shape)


Extracting Features for Distractor Model...
  Transforming strings...


  Computing cosine similarities...
  Computing character match and frequency...


Feature extraction complete. Shape: (24000, 3)


In [5]:
print("Training Random Forest Distractor Ranker...")

rf = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42)
rf.fit(X_features, y_dist)

y_pred = rf.predict(X_features)
print(classification_report(y_dist, y_pred, target_names=["Bad Distractor", "Good Distractor"]))

model_path = MODELS_DIR / "module_b_rf.pkl"
joblib.dump(rf, model_path)
print(f"Saved Distractor Ranker to {model_path}")


Training Random Forest Distractor Ranker...


                 precision    recall  f1-score   support

 Bad Distractor       0.92      0.87      0.89     15000
Good Distractor       0.80      0.87      0.83      9000

       accuracy                           0.87     24000
      macro avg       0.86      0.87      0.86     24000
   weighted avg       0.87      0.87      0.87     24000

Saved Distractor Ranker to ..\models_new\module_b_rf.pkl
